In [1]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

### Data Transformation
- Split between Training, Validation, and Testing by whole years
- Metrics to Measure: (MAE, RMSE, Mean error, Baseline_skill)
- Raw GHCND to Clean Data
    - Temporal Features (Sine Wave)
    - TMAX
    - Running Mean for TMAX
    - Running Variance for TMAX
    - TMIN
    - Running Mean for TMIN
    - Running Variance for TMIN

In [6]:
raw_ghcnd = pd.read_csv(r'../data/raw/GHCND_1980_to_2026.csv')
raw_ghcnd["DATE"] = pd.to_datetime(raw_ghcnd["DATE"])

C:\Users\Moral\AppData\Local\Temp\ipykernel_52660\781470718.py:1: DtypeWarning: Columns (7: ACMH_ATTRIBUTES, 9: ACSH_ATTRIBUTES, 13: FMTM_ATTRIBUTES, 19: PSUN_ATTRIBUTES, 25: TAVG_ATTRIBUTES, 31: TSUN_ATTRIBUTES, 33: WDF1_ATTRIBUTES, 39: WDFG_ATTRIBUTES, 41: WESD_ATTRIBUTES, 43: WSF1_ATTRIBUTES, 49: WSFG_ATTRIBUTES, 57: WT04_ATTRIBUTES, 59: WT05_ATTRIBUTES, 61: WT06_ATTRIBUTES, 63: WT07_ATTRIBUTES, 67: WT09_ATTRIBUTES, 69: WT10_ATTRIBUTES, 71: WT11_ATTRIBUTES, 73: WT13_ATTRIBUTES, 75: WT14_ATTRIBUTES, 77: WT15_ATTRIBUTES, 79: WT16_ATTRIBUTES, 81: WT18_ATTRIBUTES, 83: WT21_ATTRIBUTES, 85: WV03_ATTRIBUTES) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_ghcnd = pd.read_csv(r'../data/raw/GHCND_1980_to_2026.csv')


In [13]:
raw_ghcnd[["DATE", "TMAX", "TMIN", "PRCP"]].head()

,DATE,TMAX,TMIN,PRCP
0,1980-01-01,60.0,44.0,0.00
1,1980-01-02,61.0,37.0,0.00
2,1980-01-03,67.0,35.0,0.00
3,1980-01-04,66.0,51.0,0.11
4,1980-01-05,60.0,44.0,0.00


In [15]:
raw_ghcnd["MONTH"] = raw_ghcnd["DATE"].map(
    lambda x: x.month
)
raw_ghcnd["DAY"] = raw_ghcnd["DATE"].map(
    lambda x: x.day
)
raw_ghcnd["YEAR"] = raw_ghcnd["DATE"].map(
    lambda x: x.year
)

In [18]:
raw_ghcnd[["DATE", "TMAX", "TMIN", "PRCP"]].describe()

,DATE,TMAX,TMIN,PRCP
count,17093,17091.000000,17090.000000,17093.000000
mean,2003-05-26 00:00:00,82.536832,65.429374,0.133930
min,1980-01-01 00:00:00,38.000000,19.000000,0.000000
25%,1991-09-13 00:00:00,77.000000,58.000000,0.000000
50%,2003-05-26 00:00:00,85.000000,68.000000,0.000000
75%,2015-02-05 00:00:00,90.000000,74.000000,0.030000
max,2026-10-18 00:00:00,100.000000,85.000000,11.430000
std,NaN,9.084950,10.963633,0.422862


In [27]:
train_df = raw_ghcnd[(1980 <= raw_ghcnd.YEAR) & (raw_ghcnd.YEAR <= 2011)][["DATE", "TMAX", "TMIN", "PRCP"]]
val_df = raw_ghcnd[(2012 <= raw_ghcnd.YEAR) & (raw_ghcnd.YEAR <= 2018)][["DATE", "TMAX", "TMIN", "PRCP"]]
test_df = raw_ghcnd[(2019 <= raw_ghcnd.YEAR) & (raw_ghcnd.YEAR <= 2025)][["DATE", "TMAX", "TMIN", "PRCP"]]

In [30]:
train_df[["DATE_lag1", "TMAX_lag1", "TMIN_lag1", "PRCP_lag1"]] = train_df[["DATE", "TMAX", "TMIN", "PRCP"]].shift(1)
train_df[["DATE_lag2", "TMAX_lag2", "TMIN_lag2", "PRCP_lag2"]] = train_df[["DATE", "TMAX", "TMIN", "PRCP"]].shift(2)
train_df[["DATE_lag3", "TMAX_lag3", "TMIN_lag3", "PRCP_lag3"]] = train_df[["DATE", "TMAX", "TMIN", "PRCP"]].shift(3)
train_df[["DATE_lag4", "TMAX_lag4", "TMIN_lag4", "PRCP_lag4"]] = train_df[["DATE", "TMAX", "TMIN", "PRCP"]].shift(4)
train_df[["DATE_lag5", "TMAX_lag5", "TMIN_lag5", "PRCP_lag5"]] = train_df[["DATE", "TMAX", "TMIN", "PRCP"]].shift(5)
train_df[["DATE_lag6", "TMAX_lag6", "TMIN_lag6", "PRCP_lag6"]] = train_df[["DATE", "TMAX", "TMIN", "PRCP"]].shift(6)

In [33]:
train_df[6:].info()

<class 'pandas.DataFrame'>
RangeIndex: 11682 entries, 6 to 11687
Data columns (total 28 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   DATE       11682 non-null  datetime64[us]
 1   TMAX       11682 non-null  float64       
 2   TMIN       11682 non-null  float64       
 3   PRCP       11682 non-null  float64       
 4   DATE_lag1  11682 non-null  datetime64[us]
 5   TMAX_lag1  11682 non-null  float64       
 6   TMIN_lag1  11682 non-null  float64       
 7   PRCP_lag1  11682 non-null  float64       
 8   DATE_lag2  11682 non-null  datetime64[us]
 9   TMAX_lag2  11682 non-null  float64       
 10  TMIN_lag2  11682 non-null  float64       
 11  PRCP_lag2  11682 non-null  float64       
 12  DATE_lag3  11682 non-null  datetime64[us]
 13  TMAX_lag3  11682 non-null  float64       
 14  TMIN_lag3  11682 non-null  float64       
 15  PRCP_lag3  11682 non-null  float64       
 16  DATE_lag4  11682 non-null  datetime64[us]
 17  TMAX

### Baseline Models

#### Persistence

In [ ]:
class PersistenceBaseline():
    def __init__(self):
        pass

#### Climatological

In [ ]:
class ClimatologicalBaseline():
    def __init__(self):
        pass


### ML Models

#### Linear Regression

#### RNN

#### XGBoost